# Beyond ARIMA: Advanced Time Series Methods

*Part 7 of 8: GARCH, VAR, State Space Models, and Prophet*

Three analysts. Three unsolved problems. ARIMA couldn't help.

![Advanced Methods](advanced_methods_comparison.png)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from arch import arch_model
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 8)
np.random.seed(42)

## Part 1: GARCH - Modeling Volatility

![GARCH Concept](garch_concept_explained.png)

**Tom's problem**: Stock returns have volatility clustering.

ARIMA models the mean. GARCH models the variance!

In [ ]:
# Simulate GARCH(1,1) data
n = 1000
omega = 0.1
alpha = 0.3  # ARCH effect
beta = 0.6   # GARCH effect

returns = np.zeros(n)
variance = np.zeros(n)
variance[0] = omega / (1 - alpha - beta)

for t in range(1, n):
    variance[t] = omega + alpha * returns[t-1]**2 + beta * variance[t-1]
    returns[t] = np.sqrt(variance[t]) * np.random.normal(0, 1)

dates = pd.date_range('2020-01-01', periods=n, freq='D')
df_garch = pd.DataFrame({'returns': returns, 'variance': variance}, index=dates)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].plot(df_garch.index, df_garch['returns'], linewidth=1, alpha=0.8)
axes[0].set_title('Returns: Notice Volatility Clustering', fontsize=13, fontweight='bold')
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.3)

axes[1].plot(df_garch.index, df_garch['variance'], linewidth=1.5, color='orange')
axes[1].set_title('Conditional Variance σ²(t)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("Notice: Big moves cluster together!")

In [ ]:
# Fit GARCH(1,1)
returns_scaled = df_garch['returns'] * 100

model = arch_model(returns_scaled, vol='Garch', p=1, q=1)
model_fit = model.fit(disp='off')

print("\n✅ GARCH(1,1) fitted!")
print("\nParameters:")
print(model_fit.params)

# Forecast volatility
forecasts = model_fit.forecast(horizon=30)
forecast_variance = forecasts.variance.values[-1, :]

print(f"\n📊 30-day volatility forecast generated")
print(f"Average forecast volatility: {np.sqrt(forecast_variance).mean():.4f}")

## Part 2: VAR - Multiple Related Series

![VAR Concept](var_concept_explained.png)

**Lisa's problem**: GDP, unemployment, inflation, interest rates all affect each other.

ARIMA treats each separately. VAR models them together!

In [ ]:
# Generate multivariate data
n_obs = 500
data_multi = np.zeros((n_obs, 4))
data_multi[0] = [100, 5, 2, 3]  # GDP, Unemployment, Inflation, Interest

for t in range(1, n_obs):
    # GDP: influenced by unemployment (-), interest (-)
    data_multi[t, 0] = (0.9 * data_multi[t-1, 0] 
                       - 0.5 * data_multi[t-1, 1] 
                       - 0.3 * data_multi[t-1, 3]
                       + np.random.normal(0, 1))
    
    # Unemployment: influenced by GDP (-)
    data_multi[t, 1] = (-0.2 * data_multi[t-1, 0]
                       + 0.8 * data_multi[t-1, 1]
                       + np.random.normal(0, 0.3))
    
    # Inflation: influenced by GDP (+), interest (-)
    data_multi[t, 2] = (0.1 * data_multi[t-1, 0]
                       + 0.7 * data_multi[t-1, 2]
                       - 0.2 * data_multi[t-1, 3]
                       + np.random.normal(0, 0.2))
    
    # Interest: influenced by inflation (+)
    data_multi[t, 3] = (0.5 * data_multi[t-1, 2]
                       + 0.6 * data_multi[t-1, 3]
                       + np.random.normal(0, 0.2))

dates_multi = pd.date_range('2000-01-01', periods=n_obs, freq='Q')
df_multi = pd.DataFrame(
    data_multi,
    index=dates_multi,
    columns=['GDP', 'Unemployment', 'Inflation', 'Interest_Rate']
)

print("Multivariate data generated")
print(f"Shape: {df_multi.shape}")
print("\nCorrelations:")
print(df_multi.corr())

In [ ]:
# Fit VAR model
model_var = VAR(df_multi)
results = model_var.fit(maxlags=5, ic='aic')

print(f"\n✅ VAR model fitted!")
print(f"Optimal lag order: {results.k_ar}")
print("\nModel summary:")
print(results.summary())

In [ ]:
# Granger Causality Test
print("\nGranger Causality Tests:")
print("\nDoes GDP Granger-cause Unemployment?")
gc_result = grangercausalitytests(df_multi[['Unemployment', 'GDP']], 
                                   maxlag=4, verbose=False)

for lag in range(1, 5):
    p_value = gc_result[lag][0]['ssr_ftest'][1]
    print(f"  Lag {lag}: p={p_value:.4f} {'✓ Significant' if p_value < 0.05 else ''}")

In [ ]:
# Forecast
forecast = results.forecast(df_multi.values[-results.k_ar:], steps=20)
forecast_dates = pd.date_range(start=df_multi.index[-1] + pd.DateOffset(months=3),
                               periods=20, freq='Q')
forecast_df = pd.DataFrame(forecast, index=forecast_dates, columns=df_multi.columns)

print("\n✅ Forecast generated for next 20 quarters")
print(forecast_df.head())

## Part 3: Prophet - Production-Ready Forecasting

**Alex's problem**: Multiple seasonalities + holidays.

Prophet handles it all automatically!

In [ ]:
# Note: Prophet requires installation
# !pip install prophet

try:
    from prophet import Prophet
    
    # Generate complex seasonal data
    n_prophet = 365 * 2
    time = np.arange(n_prophet)
    trend = 100 + 0.05 * time
    weekly = 10 * np.sin(2 * np.pi * time / 7)
    yearly = 30 * np.sin(2 * np.pi * time / 365)
    noise = np.random.normal(0, 5, n_prophet)
    sales = trend + weekly + yearly + noise
    
    dates_prophet = pd.date_range('2021-01-01', periods=n_prophet, freq='D')
    df_prophet = pd.DataFrame({'ds': dates_prophet, 'y': sales})
    
    # Fit Prophet
    model_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=True)
    model_prophet.fit(df_prophet)
    
    # Forecast
    future = model_prophet.make_future_dataframe(periods=90)
    forecast_prophet = model_prophet.predict(future)
    
    print("\n✅ Prophet model fitted and forecast generated!")
    
    # Plot
    fig = model_prophet.plot(forecast_prophet, figsize=(16, 6))
    plt.title('Prophet Forecast')
    plt.show()
    
except ImportError:
    print("\n⚠️  Prophet not installed")
    print("Install with: pip install prophet")
    print("\nProphet features:")
    print("  • Automatic seasonality detection")
    print("  • Easy holiday modeling")
    print("  • Robust to outliers")
    print("  • Fast and production-ready")

## Method Selection Framework

![Decision Tree](advanced_methods_decision_tree.png)

In [ ]:
def recommend_method(characteristics):
    """Recommend time series method based on characteristics"""
    print("\n" + "="*60)
    print("METHOD RECOMMENDATION")
    print("="*60)
    
    recommendations = []
    
    if characteristics.get('volatility_clustering'):
        recommendations.append("GARCH (for volatility modeling)")
    
    if characteristics.get('multiple_series'):
        recommendations.append("VAR (for interdependent variables)")
    
    if characteristics.get('multiple_seasonalities') or characteristics.get('holidays'):
        recommendations.append("Prophet (for complex seasonality)")
    
    if characteristics.get('regime_changes'):
        recommendations.append("State Space Models")
    
    if not recommendations:
        recommendations.append("ARIMA (start simple!)")
    
    print("\n📊 Your characteristics:")
    for k, v in characteristics.items():
        print(f"  • {k}: {v}")
    
    print("\n💡 Recommended methods:")
    for i, rec in enumerate(recommendations, 1):
        print(f"  {i}. {rec}")
    
    return recommendations

# Example
example = {
    'volatility_clustering': True,
    'multiple_series': False,
    'multiple_seasonalities': False,
    'holidays': False,
    'regime_changes': False
}

recommend_method(example)

## Key Takeaways

✅ **GARCH**: Model volatility, not just mean

✅ **VAR**: Model multiple series together

✅ **State Space**: Handle regime changes

✅ **Prophet**: Fast, robust, production-ready

✅ **Choose based on problem**: No one-size-fits-all

## What's Next

**Part 8 (Finale)**: Deep Learning for Time Series
- LSTM & Transformers
- When DL beats classical
- Complete decision framework